# 01 — Exploratory Data Analysis

This notebook explores which factors are associated with football player market value before we engineer features or train a model.  

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Locate the project root from either the project or notebooks folder.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Allow this notebook to import code from src/.
sys.path.insert(0, str(PROJECT_ROOT))

from src.load import load_csv


ACCENT = "#7C3AED"
PLOT_TEMPLATE = "plotly_dark"

players = load_csv("players")

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded {len(players):,} players")
print(f"Python environment: {sys.executable}")

Project root: c:\Users\HP\pl-transfer-value
Loaded 50,149 players
Python environment: c:\Users\HP\pl-transfer-value\venv\Scripts\python.exe


In [4]:
# Convert the target to numeric values and remove missing targets.
market_values = pd.to_numeric(
    players["market_value_in_eur"],
    errors="coerce",
).dropna()

# Keep only players with a known market value.
eda_players = players.loc[market_values.index].copy()
eda_players["market_value_in_eur"] = market_values

# Create the transformed target used later during training.
eda_players["log_market_value"] = np.log1p(
    eda_players["market_value_in_eur"]
)


summary = pd.DataFrame(
    {
        "Metric": [
            "Players with values",
            "Median value",
            "90th percentile",
            "99th percentile",
            "Maximum value",
        ],
        "Result": [
            f"{len(market_values):,}",
            f"€{market_values.median() / 1_000_000:,.2f}M",
            f"€{market_values.quantile(0.90) / 1_000_000:,.2f}M",
            f"€{market_values.quantile(0.99) / 1_000_000:,.2f}M",
            f"€{market_values.max() / 1_000_000:,.2f}M",
        ],
    }
)

display(summary)

print(f"Raw-target skewness: {market_values.skew():.2f}")
print(
    "Log-target skewness: "
    f"{eda_players['log_market_value'].skew():.2f}"
)


fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=False,
    subplot_titles=(
        "Raw market value",
        "After log1p transformation",
    ),
)

fig.add_trace(
    go.Histogram(
        x=market_values / 1_000_000,
        nbinsx=60,
        marker_color=ACCENT,
        hovertemplate=(
            "Value: €%{x:.1f}M"
            "<br>Players: %{y:,}"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Histogram(
        x=eda_players["log_market_value"],
        nbinsx=60,
        marker_color=ACCENT,
        hovertemplate=(
            "log1p(value): %{x:.2f}"
            "<br>Players: %{y:,}"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.update_layout(
    template=PLOT_TEMPLATE,
    title={
        "text": "Market value is heavily right-skewed",
        "x": 0.02,
    },
    height=470,
    bargap=0.04,
    showlegend=False,
)

fig.update_xaxes(
    title_text="Market value (€ millions)",
    row=1,
    col=1,
)

fig.update_xaxes(
    title_text="log1p(market value)",
    row=1,
    col=2,
)

fig.update_yaxes(
    title_text="Number of players",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Number of players",
    row=1,
    col=2,
)

fig.show()

,Metric,Result
0,Players with values,"41,528"
1,Median value,€0.28M
2,90th percentile,€2.50M
3,99th percentile,€25.00M
4,Maximum value,€200.00M


Raw-target skewness: 11.75
Log-target skewness: 0.70


## 2. Age versus median market value

Player value is expected to follow a curved age profile rather than a straight line.

We use the **median** instead of the mean because a small number of superstar players would pull the mean upward. The median better represents a typical player at each age.

In [6]:
# Use the newest valuation date as the reference point for calculating age.
valuation_dates = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "player_valuations.csv",
    usecols=["date"],
    parse_dates=["date"],
)

REFERENCE_DATE = valuation_dates["date"].max()

# Calculate each player's approximate age on the reference date.
eda_players["date_of_birth"] = pd.to_datetime(
    eda_players["date_of_birth"],
    errors="coerce",
)

eda_players["age"] = np.floor(
    (
        REFERENCE_DATE - eda_players["date_of_birth"]
    ).dt.days / 365.25
)

# Keep a sensible professional-football age range.
age_data = eda_players[
    eda_players["age"].between(16, 40)
].copy()

age_data["age"] = age_data["age"].astype(int)

# Calculate the typical market value at each age.
age_curve = (
    age_data
    .groupby("age", as_index=False)
    .agg(
        median_value_eur=(
            "market_value_in_eur",
            "median",
        ),
        player_count=(
            "player_id",
            "nunique",
        ),
    )
)

age_curve["median_value_millions"] = (
    age_curve["median_value_eur"] / 1_000_000
)

# Find the observed peak.
peak_row = age_curve.loc[
    age_curve["median_value_eur"].idxmax()
]

print(f"Reference date: {REFERENCE_DATE.date()}")
print(
    f"Peak median value occurs at age "
    f"{int(peak_row['age'])}: "
    f"€{peak_row['median_value_millions']:.2f}M"
)

display(age_curve.head())

Reference date: 2026-06-12
Peak median value occurs at age 18: €0.40M


,age,median_value_eur,player_count,median_value_millions
0,16,300000.0,21,0.300
1,17,375000.0,100,0.375
2,18,400000.0,389,0.400
3,19,300000.0,773,0.300
4,20,350000.0,1217,0.350


In [8]:
# Find all ages tied for the highest median value.
maximum_median = (
    age_curve["median_value_millions"].max()
)

peak_ages = age_curve.loc[
    np.isclose(
        age_curve["median_value_millions"],
        maximum_median,
    ),
    "age",
].tolist()

print(
    "Highest median market value: "
    f"€{maximum_median:.2f}M"
)

print(
    "Ages sharing the highest median: "
    f"{peak_ages}"
)


# Create the corrected age-profile chart.
fig = go.Figure()

# Highlight the prime-age region.
fig.add_vrect(
    x0=22.5,
    x1=28.5,
    fillcolor=ACCENT,
    opacity=0.12,
    line_width=0,
    annotation_text="Prime-age window",
    annotation_position="top left",
)

fig.add_trace(
    go.Scatter(
        x=age_curve["age"],
        y=age_curve["median_value_millions"],
        mode="lines+markers",
        line={
            "color": ACCENT,
            "width": 3,
        },
        marker={
            "size": 8,
            "color": ACCENT,
        },
        customdata=age_curve[["player_count"]],
        hovertemplate=(
            "Age: %{x}"
            "<br>Median value: €%{y:.2f}M"
            "<br>Players: %{customdata[0]:,}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    template=PLOT_TEMPLATE,
    title={
        "text": (
            "Prime-age values plateau before "
            "declining from age 30"
        ),
        "x": 0.02,
    },
    xaxis_title="Player age",
    yaxis_title="Median market value (€ millions)",
    height=500,
    showlegend=False,
    hovermode="x unified",
)

fig.update_xaxes(
    dtick=2,
    range=[15.5, 40.5],
)

fig.update_yaxes(
    rangemode="tozero",
    ticksuffix="M",
)

fig.show()

Highest median market value: €0.40M
Ages sharing the highest median: [18, 23, 24, 25, 26, 27, 28, 29]


### Interpretation

Median market value remains relatively stable throughout the 23–29 prime-age period before declining noticeably from age 30. The highest median is shared by several ages rather than belonging to one unique peak.

This demonstrates that age has a non-linear relationship with market value. Adding one year does not have the same effect on a 20-year-old, a 27-year-old, and a 35-year-old. We will therefore include both `age` and `age_squared` in the model.

The relatively high value among very young players may also reflect selection bias: teenagers appearing in professional football data are more likely to be unusually promising prospects.

## 3. Market value by position

This chart compares the median market value of players in each broad position group.

We use the median because superstar players can heavily distort the mean. This is an unconditional comparison: differences may also reflect age, league, club quality, minutes, and performance. The model will later account for these features together.

In [10]:
# ============================================================
# MARKET VALUE BY POSITION
# ============================================================

# Keep the columns required for this comparison.
position_data = eda_players[
    [
        "player_id",
        "position",
        "market_value_in_eur",
    ]
].dropna(
    subset=[
        "position",
        "market_value_in_eur",
    ]
)
# "Missing" is a dataset label, not a real position.
position_data = position_data[
    position_data["position"] != "Missing"
].copy()


# Calculate the median value and number of players
# in each broad position group.
position_summary = (
    position_data
    .groupby("position", as_index=False)
    .agg(
        median_value_eur=(
            "market_value_in_eur",
            "median",
        ),
        player_count=(
            "player_id",
            "nunique",
        ),
    )
)

position_summary["median_value_millions"] = (
    position_summary["median_value_eur"]
    / 1_000_000
)

# Rank positions from highest to lowest median value.
position_summary = position_summary.sort_values(
    "median_value_millions",
    ascending=False,
).reset_index(drop=True)


# Print the exact results.
display(
    position_summary[
        [
            "position",
            "median_value_millions",
            "player_count",
        ]
    ]
)


# ------------------------------------------------------------
# Create the chart
# ------------------------------------------------------------

fig = go.Figure(
    go.Bar(
        x=position_summary["position"],
        y=position_summary[
            "median_value_millions"
        ],
        marker_color=ACCENT,
        text=[
            f"€{value:.2f}M"
            for value in position_summary[
                "median_value_millions"
            ]
        ],
        customdata=position_summary[
            ["player_count"]
        ],
        hovertemplate=(
            "Position: %{x}"
            "<br>Median value: €%{y:.2f}M"
            "<br>Players: %{customdata[0]:,}"
            "<extra></extra>"
        ),
    )
)

fig.update_traces(
    textposition="outside",
    cliponaxis=False,
)

fig.update_layout(
    template=PLOT_TEMPLATE,
    title={
        "text": (
            "Median market value differs "
            "across positions"
        ),
        "x": 0.02,
    },
    xaxis_title="Position",
    yaxis_title=(
        "Median market value (€ millions)"
    ),
    height=500,
    showlegend=False,
    bargap=0.25,
)

fig.update_yaxes(
    rangemode="tozero",
    tickprefix="€",
    ticksuffix="M",
)

fig.show()

,position,median_value_millions,player_count
0,Attack,0.30,11369
1,Defender,0.30,13411
2,Midfield,0.30,12015
3,Goalkeeper,0.15,4604


### Interpretation

Attackers, midfielders, and defenders share the same €0.30M median market value in the full dataset, while goalkeepers have a lower median of €0.15M.

The tied outfield medians do not prove that position is irrelevant. Transfermarkt values are rounded, and this raw comparison mixes players from different ages, leagues, clubs, and performance levels. Position may become more informative after controlling for those factors.

Position is especially important when interpreting performance statistics. For example, zero goals is normal for a goalkeeper but may be concerning for a striker. We will therefore one-hot encode position and later investigate position-aware relationships.

### Interpretation

Attackers, midfielders, and defenders share the same €0.30M median market value in the full dataset, while goalkeepers have a lower median of €0.15M.

The tied outfield medians do not prove that position is irrelevant. Transfermarkt values are rounded, and this raw comparison mixes players from different ages, leagues, clubs, and performance levels. Position may become more informative after controlling for those factors.

Position is especially important when interpreting performance statistics. For example, zero goals is normal for a goalkeeper but may be concerning for a striker. We will therefore one-hot encode position and later investigate position-aware relationships.

In [11]:
# ============================================================
# GOALS VERSUS MARKET VALUE BY POSITION
# ============================================================

# Load only the appearance columns required for this chart.
# This uses less memory than loading all 13 columns.
appearance_data = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "raw"
    / "appearances.csv",
    usecols=[
        "game_id",
        "player_id",
        "date",
        "goals",
        "assists",
        "minutes_played",
    ],
    parse_dates=["date"],
)


# ------------------------------------------------------------
# Define the latest European-season analysis window
# ------------------------------------------------------------

latest_appearance_date = appearance_data["date"].max()

# European football seasons normally begin around July.
if latest_appearance_date.month >= 7:
    season_start_year = latest_appearance_date.year
else:
    season_start_year = (
        latest_appearance_date.year - 1
    )

season_start_date = pd.Timestamp(
    year=season_start_year,
    month=7,
    day=1,
)

season_end_date = latest_appearance_date

season_label = (
    f"{season_start_year}/"
    f"{str(season_start_year + 1)[-2:]}"
)

print(f"Analysis window: {season_label}")
print(
    f"Dates: {season_start_date.date()} "
    f"to {season_end_date.date()}"
)


# Keep appearances inside the latest season window.
latest_season_appearances = appearance_data[
    appearance_data["date"].between(
        season_start_date,
        season_end_date,
    )
].copy()


# ------------------------------------------------------------
# Aggregate matches into one row per player
# ------------------------------------------------------------

season_stats = (
    latest_season_appearances
    .groupby("player_id", as_index=False)
    .agg(
        goals=("goals", "sum"),
        assists=("assists", "sum"),
        minutes_played=(
            "minutes_played",
            "sum",
        ),
        appearances=("game_id", "nunique"),
    )
)

print(
    "Players before the minutes filter: "
    f"{len(season_stats):,}"
)

# Remove players with tiny, unreliable samples.
season_stats = season_stats[
    season_stats["minutes_played"] >= 300
].copy()

print(
    "Players with at least 300 minutes: "
    f"{len(season_stats):,}"
)


# ------------------------------------------------------------
# Align market values with the end of the season
# ------------------------------------------------------------

valuation_data = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "raw"
    / "player_valuations.csv",
    usecols=[
        "player_id",
        "date",
        "market_value_in_eur",
    ],
    parse_dates=["date"],
)

# Do not use a valuation recorded after the analysis window.
valuation_data = valuation_data[
    valuation_data["date"] <= season_end_date
].copy()

# Select the most recent available valuation for each player.
latest_player_values = (
    valuation_data
    .sort_values(
        ["player_id", "date"]
    )
    .groupby(
        "player_id",
        as_index=False,
    )
    .tail(1)
    .rename(
        columns={
            "date": "valuation_date",
        }
    )
)


# ------------------------------------------------------------
# Add player names, positions, and aligned values
# ------------------------------------------------------------

player_metadata = (
    players[
        [
            "player_id",
            "name",
            "position",
        ]
    ]
    .drop_duplicates("player_id")
)

scatter_data = (
    season_stats
    .merge(
        player_metadata,
        on="player_id",
        how="inner",
    )
    .merge(
        latest_player_values,
        on="player_id",
        how="inner",
    )
)

valid_positions = [
    "Attack",
    "Midfield",
    "Defender",
    "Goalkeeper",
]

scatter_data = scatter_data[
    scatter_data["position"].isin(
        valid_positions
    )
    & (
        scatter_data["market_value_in_eur"]
        > 0
    )
].copy()

scatter_data["market_value_millions"] = (
    scatter_data["market_value_in_eur"]
    / 1_000_000
)

print(
    "Players in the final chart: "
    f"{len(scatter_data):,}"
)


# Show how many players appear in each position group.
position_counts = (
    scatter_data
    .groupby("position", as_index=False)
    .agg(
        players=("player_id", "nunique"),
        median_goals=("goals", "median"),
        median_value_millions=(
            "market_value_millions",
            "median",
        ),
    )
    .sort_values(
        "median_value_millions",
        ascending=False,
    )
)

display(position_counts)


# ------------------------------------------------------------
# Create the scatter plot
# ------------------------------------------------------------

position_colors = {
    "Attack": "#7C3AED",
    "Midfield": "#22C55E",
    "Defender": "#38BDF8",
    "Goalkeeper": "#F59E0B",
}

fig = go.Figure()

for position in valid_positions:
    position_frame = scatter_data[
        scatter_data["position"] == position
    ]

    fig.add_trace(
        go.Scattergl(
            x=position_frame["goals"],
            y=position_frame[
                "market_value_millions"
            ],
            mode="markers",
            name=position,
            marker={
                "color": position_colors[position],
                "size": 7,
                "opacity": 0.55,
            },
            customdata=position_frame[
                [
                    "name",
                    "assists",
                    "minutes_played",
                    "appearances",
                ]
            ].to_numpy(),
            hovertemplate=(
                "<b>%{customdata[0]}</b>"
                f"<br>Position: {position}"
                "<br>Goals: %{x}"
                "<br>Assists: %{customdata[1]}"
                "<br>Minutes: %{customdata[2]:,}"
                "<br>Appearances: %{customdata[3]}"
                "<br>Market value: €%{y:.2f}M"
                "<extra></extra>"
            ),
        )
    )


# ------------------------------------------------------------
# Style the chart
# ------------------------------------------------------------

fig.update_layout(
    template=PLOT_TEMPLATE,
    title={
        "text": (
            f"Goals have different meanings "
            f"across positions — {season_label}"
        ),
        "x": 0.02,
    },
    xaxis_title="Goals during the season",
    yaxis_title=(
        "Market value (€ millions, log scale)"
    ),
    height=650,
    legend_title_text="Position",
    hovermode="closest",
)

fig.update_xaxes(
    rangemode="tozero",
    dtick=5,
)

# The visual log scale prevents €100M players from
# compressing most players against the bottom.
fig.update_yaxes(
    type="log",
)

fig.show()

Analysis window: 2025/26
Dates: 2025-07-01 to 2026-06-28
Players before the minutes filter: 9,051
Players with at least 300 minutes: 6,037
Players in the final chart: 5,968


,position,players,median_goals,median_value_millions
0,Attack,1640,3.0,2.5
3,Midfield,1742,1.0,2.2
1,Defender,2092,0.0,2.0
2,Goalkeeper,494,0.0,1.2


### Interpretation

Among players with at least 300 minutes in the 2025/26 analysis window, attackers recorded a median of three goals, midfielders one, and defenders and goalkeepers zero. Despite both having zero median goals, defenders and goalkeepers still retained substantial market values.

This shows why goals cannot be interpreted identically across positions. Zero goals is expected for a goalkeeper or defender but may carry more information for an attacker. A single model that treats one additional goal as having the same meaning for every player would miss this relationship.

Position will therefore be one-hot encoded, and later modelling can include position-aware interactions or separate position models.

The market-value axis is logarithmic. Equal vertical distances represent multiplicative changes, allowing €1M, €10M, and €100M players to remain visible on the same chart.

## 5. Contract years remaining versus market value

A club normally has more negotiating power when a player has several years left on their contract. As expiration approaches, the player may become available for a reduced fee or eventually leave as a free agent.

This is a current-snapshot analysis. The dataset does not contain reliable historical contract-expiration dates, so current contract information must not be attached to earlier seasons because that would leak future information.

In [13]:
# ============================================================
# CONTRACT YEARS REMAINING VERSUS MARKET VALUE
# ============================================================

# Use only players with a market value and contract date.
contract_data = eda_players[
    [
        "player_id",
        "name",
        "contract_expiration_date",
        "market_value_in_eur",
    ]
].copy()

contract_data["contract_expiration_date"] = (
    pd.to_datetime(
        contract_data["contract_expiration_date"],
        errors="coerce",
    )
)

players_before_contract_filter = len(
    contract_data
)

contract_data = contract_data.dropna(
    subset=[
        "contract_expiration_date",
        "market_value_in_eur",
    ]
).copy()

print(
    "Players with market values: "
    f"{players_before_contract_filter:,}"
)

print(
    "Players with known contract dates: "
    f"{len(contract_data):,}"
)


# ------------------------------------------------------------
# Calculate years remaining
# ------------------------------------------------------------

CONTRACT_REFERENCE_DATE = REFERENCE_DATE

contract_data["contract_years_remaining"] = (
    (
        contract_data["contract_expiration_date"]
        - CONTRACT_REFERENCE_DATE
    ).dt.days
    / 365.25
)

# Keep contracts that had not expired on the reference date.
contract_data = contract_data[
    contract_data["contract_years_remaining"] >= 0
].copy()

print(
    "Players with known, non-expired contracts: "
    f"{len(contract_data):,}"
)

# ------------------------------------------------------------
# Create ordered contract-time groups
# ------------------------------------------------------------

contract_labels = [
    "< 1 year",
    "1–2 years",
    "2–3 years",
    "3–4 years",
    "4+ years",
]

contract_data["contract_group"] = pd.cut(
    contract_data["contract_years_remaining"],
    bins=[
        0,
        1,
        2,
        3,
        4,
        np.inf,
    ],
    labels=contract_labels,
    right=False,
    include_lowest=True,
    ordered=True,
)


# ------------------------------------------------------------
# Summarize market value within each contract group
# ------------------------------------------------------------

contract_summary = (
    contract_data
    .groupby(
        "contract_group",
        observed=True,
        as_index=False,
    )
    .agg(
        median_value_eur=(
            "market_value_in_eur",
            "median",
        ),
        lower_quartile_eur=(
            "market_value_in_eur",
            lambda values: values.quantile(0.25),
        ),
        upper_quartile_eur=(
            "market_value_in_eur",
            lambda values: values.quantile(0.75),
        ),
        player_count=(
            "player_id",
            "nunique",
        ),
    )
)

contract_summary["median_value_millions"] = (
    contract_summary["median_value_eur"]
    / 1_000_000
)

contract_summary["lower_quartile_millions"] = (
    contract_summary["lower_quartile_eur"]
    / 1_000_000
)

contract_summary["upper_quartile_millions"] = (
    contract_summary["upper_quartile_eur"]
    / 1_000_000
)

display(
    contract_summary[
        [
            "contract_group",
            "median_value_millions",
            "lower_quartile_millions",
            "upper_quartile_millions",
            "player_count",
        ]
    ]
)


# ------------------------------------------------------------
# Create the ordered line chart
# ------------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=contract_summary["contract_group"],
        y=contract_summary[
            "median_value_millions"
        ],
        mode="lines+markers",
        line={
            "color": ACCENT,
            "width": 3,
        },
        marker={
            "color": ACCENT,
            "size": 11,
        },
        customdata=contract_summary[
            [
                "lower_quartile_millions",
                "upper_quartile_millions",
                "player_count",
            ]
        ].to_numpy(),
        hovertemplate=(
            "Contract remaining: %{x}"
            "<br>Median value: €%{y:.2f}M"
            "<br>25th percentile: "
            "€%{customdata[0]:.2f}M"
            "<br>75th percentile: "
            "€%{customdata[1]:.2f}M"
            "<br>Players: %{customdata[2]:,}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    template=PLOT_TEMPLATE,
    title={
        "text": (
            "Market value by contract "
            "time remaining"
        ),
        "x": 0.02,
    },
    xaxis_title="Contract time remaining",
    yaxis_title=(
        "Median market value (€ millions)"
    ),
    height=520,
    showlegend=False,
)

fig.update_xaxes(
    categoryorder="array",
    categoryarray=contract_labels,
)

fig.update_yaxes(
    rangemode="tozero",
    tickprefix="€",
    ticksuffix="M",
)

fig.show()

Players with market values: 41,528
Players with known contract dates: 26,874
Players with known, non-expired contracts: 17,306


,contract_group,median_value_millions,lower_quartile_millions,upper_quartile_millions,player_count
0,< 1 year,0.3,0.15000,0.7,4557
1,1–2 years,0.5,0.25000,1.5,5190
2,2–3 years,0.9,0.39375,2.5,3976
3,3–4 years,1.8,0.60000,6.0,2323
4,4+ years,5.5,1.50000,18.0,1260


### Interpretation

Market value rises sharply with contract time remaining. Players with less than one year remaining have a median value of €0.3M, compared with €5.5M for players with at least four years remaining—approximately an eighteen-fold difference.

A longer contract gives the selling club greater negotiating power, while players approaching expiration may become available at a reduced price or eventually leave without a transfer fee. Contract duration is therefore potentially one of the strongest non-performance valuation features.

However, this chart shows association rather than pure causation. Clubs are also more likely to give long contracts to young, talented, and already valuable players. The model must consider contract duration together with age, performance, league, and club strength.

Only 17,306 players have both a usable market value and a non-expired contract date. Furthermore, the dataset contains current contract dates rather than reliable historical contract records. Using these current dates for earlier seasons would leak future information, so the historical model must handle this limitation honestly.